# Bi-Directional Encoder Representation from transformer

In [2]:
!pip install tf-keras

In [22]:
from transformers import BertForMaskedLM, BertTokenizer
import tensorflow as tf
import torch

### Load the tokenizer and Model

In [23]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForMaskedLM.from_pretrained('bert-base-uncased')


Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Input sentence with proper mask token

In [5]:
input_text = f"the cat sat on the{tokenizer.mask_token} and looked out the window"

In [6]:
# tokenize

inputs = tokenizer(input_text, return_tensors='pt')
inputs

{'input_ids': tensor([[ 101, 1996, 4937, 2938, 2006, 1996,  103, 1998, 2246, 2041, 1996, 3332,
          102]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}

# Run model

In [7]:
outputs = model(**inputs)
logits = outputs.logits


In [8]:
import torch

In [9]:
mask_token_index = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1].item()

predicted_token_id = logits[0, mask_token_index].detach().argmax().item()
predicted_token = tokenizer.decode([predicted_token_id])

print(predicted_token)

bed


In [10]:
import torch

top_k = 5

# Detach gradients and compute top-k directly in PyTorch
mask_token_index = (inputs["input_ids"] == tokenizer.mask_token_id).nonzero(as_tuple=True)[1]
top_k_ids = torch.topk(logits[0, mask_token_index], k=top_k).indices.squeeze().tolist()

# Decode the tokens
top_k_tokens = [tokenizer.decode([token_id]).strip() for token_id in top_k_ids]

print(top_k_tokens)

['bed', 'floor', 'couch', 'sofa', 'porch']


In [11]:
print("Top predictions:")
for i, token in enumerate(top_k_tokens):
    print(f"{i + 1}. {token}")

Top predictions:
1. bed
2. floor
3. couch
4. sofa
5. porch


# Sentiment analysis

In [12]:
! pip install -q tensorflow transformers datasets

In [14]:
#!pip install --upgrade datasets huggingface_hub

## Load a small IMBD Data

## Load tokenizer & model

In [3]:
import torch
from datasets import load_dataset
from transformers import (
    BertForMaskedLM,
    BertTokenizer,
    DistilBertForSequenceClassification,
    DistilBertTokenizer,
    Trainer,
    TrainingArguments,
)

In [4]:
tokenizer_mlm = BertTokenizer.from_pretrained("bert-base-uncased")
model_mlm = BertForMaskedLM.from_pretrained("bert-base-uncased")

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
input_text = f"the cat sat on the {tokenizer_mlm.mask_token} and looked out the window"
inputs = tokenizer_mlm(input_text, return_tensors="pt")

In [6]:
with torch.no_grad():
  outputs = model_mlm(**inputs)
  logits = outputs.logits

mask_token_index = (
    (inputs["input_ids"] == tokenizer_mlm.mask_token_id)
    .nonzero(as_tuple=True)[1]
    .item()
)
top_k = 5
top_k_ids = (
    torch.topk(logits[0, mask_token_index], k=top_k).indices.squeeze().tolist()
)
top_k_tokens = [
    tokenizer_mlm.decode([token_id]).strip() for token_id in top_k_ids
]

print("Top predictions:")
for i, token in enumerate(top_k_tokens):
  print(f"{i + 1}. {token}")

Top predictions:
1. bed
2. floor
3. couch
4. sofa
5. porch


## Sentiment anaylis

In [7]:
dataset = load_dataset("stanfordnlp/imdb")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [8]:
small_train = dataset["train"].shuffle(seed=42).select(range(500))
small_test = dataset["test"].shuffle(seed=42).select(range(100))

In [9]:
tokenizer_clf = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")
model_clf = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
def tokenize(batch):
  return tokenizer_clf(
      batch["text"], padding="max_length", truncation=True, max_length=128
  )

In [11]:
small_train = small_train.map(tokenize, batched=True)
small_test = small_test.map(tokenize, batched=True)

In [12]:
small_train = small_train.rename_column("label", "labels")
small_test = small_test.rename_column("label", "labels")

In [13]:
small_train.set_format(
    type="torch", columns=["input_ids", "attention_mask", "labels"]
)
small_test.set_format(
    type="torch", columns=["input_ids", "attention_mask", "labels"]
)

In [14]:
# pip install -U accelerate transformers

In [16]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    learning_rate=2e-5,
    logging_steps=10,
    save_strategy="no",
)

trainer = Trainer(
    model=model_clf,
    args=training_args,
    train_dataset=small_train,
    eval_dataset=small_test,
)

trainer.train()

Epoch,Training Loss,Validation Loss
1,0.625248,0.576395
2,0.393502,0.422357
3,0.319209,0.410787


TrainOutput(global_step=96, training_loss=0.47292187809944153, metrics={'train_runtime': 18.6411, 'train_samples_per_second': 80.467, 'train_steps_per_second': 5.15, 'total_flos': 49675274496000.0, 'train_loss': 0.47292187809944153, 'epoch': 3.0})

In [17]:
import torch


def predict_sentiment(text):
  # 1. Tokenize the input text
  inputs = tokenizer_clf(
      text, return_tensors="pt", truncation=True, padding=True, max_length=128
  )

  # Move inputs to the same device as the model (CPU or GPU)
  device = model_clf.device
  inputs = {k: v.to(device) for k, v in inputs.items()}

  # 2. Forward pass with no gradient calculation
  model_clf.eval()
  with torch.no_grad():
    outputs = model_clf(**inputs)
    logits = outputs.logits

  # 3. Apply softmax to get probabilities
  probabilities = torch.softmax(logits, dim=-1)

  # 4. Get the predicted class and confidence score
  pred_class = torch.argmax(probabilities, dim=-1).item()
  confidence = probabilities[0][pred_class].item()

  # Map IMDB class indices (0: Negative, 1: Positive)
  label_map = {0: "Negative", 1: "Positive"}
  sentiment = label_map[pred_class]

  return sentiment, f"{confidence:.4f}"

In [18]:
examples = [
    "This movie was amazing! i loved every movement.",
    "Worst movie ever, Total waste of time.",
    "It was okay, not great but not terrible either.",
]

for example in examples:
  sentiment, confidence = predict_sentiment(example)
  print(f"{example}: {sentiment} (confidence: {confidence})")

This movie was amazing! i loved every movement.: Positive (confidence: 0.8781)
Worst movie ever, Total waste of time.: Negative (confidence: 0.8153)
It was okay, not great but not terrible either.: Positive (confidence: 0.6651)
